# Analisis Exploratorio: Inflacion en Ecuador y economias de referencia (2014-2024)

Agente de Analisis Economico. Fuente: `data/processed/inflacion_pib_desempleo.csv` (Banco Mundial, consultado 2026-07-16).

Este notebook es exploratorio: reproduce en un entorno interactivo lo que `scripts/calculate_indicators.py` calcula de forma automatizada, para que el equipo pueda inspeccionar los datos paso a paso.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/processed/inflacion_pib_desempleo.csv')
df.head(10)

In [ ]:
df['indicador'].value_counts()

## Evolucion de la inflacion por pais

In [ ]:
inflacion = df[df['indicador'] == 'inflacion_precios_consumidor']
pivot = inflacion.pivot(index='anio', columns='pais', values='valor')

ax = pivot.plot(figsize=(9, 5), marker='o')
ax.set_title('Inflacion anual (% precios al consumidor), 2014-2024')
ax.set_xlabel('Ano')
ax.set_ylabel('Inflacion (%)')
ax.axhline(0, color='gray', linewidth=0.8)
plt.tight_layout()
plt.savefig('../outputs/charts/evolucion_inflacion.png', dpi=150)
plt.show()

## Estadistica descriptiva rapida

In [ ]:
inflacion.groupby('pais')['valor'].describe().round(2).sort_values('mean')

**Interpretacion:** Panama y Ecuador -las dos economias dolarizadas de la muestra- presentan la
menor inflacion promedio y la menor dispersion del grupo. El detalle formal de estos indicadores,
con redondeo e interpretacion economica completa, se encuentra en `outputs/tables/indicadores_comparativos.csv`
y en `docs/informe_final.pdf` (seccion 10.1).

## Relacion entre inflacion y crecimiento del PIB (vista rapida)

In [ ]:
pib = df[df['indicador'] == 'crecimiento_pib'][['pais', 'anio', 'valor']].rename(columns={'valor': 'pib'})
infl = inflacion[['pais', 'anio', 'valor']].rename(columns={'valor': 'inflacion'})
merged = infl.merge(pib, on=['pais', 'anio'])

fig, ax = plt.subplots(figsize=(7, 6))
for pais, group in merged.groupby('pais'):
    ax.scatter(group['inflacion'], group['pib'], label=pais, alpha=0.8)
ax.set_xlabel('Inflacion (%)')
ax.set_ylabel('Crecimiento del PIB (%)')
ax.set_title('Inflacion vs. crecimiento del PIB, 2014-2024')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/charts/inflacion_vs_pib.png', dpi=150)
plt.show()

No se observa un patron lineal unico y claro entre ambas variables a simple vista; el analisis
formal de correlacion por pais (con p-valores) se realiza en `econometric_analysis.ipynb` y en
`scripts/econometric_model.py`.